# 7.0 Reports (run in SageMaker Studio)

**SHAP explainability** for the evaluated XGBoost model, then a consolidated index `monitoring/reports/index.json` of the data-quality and model-quality constraints/violations, the SHAP outputs and the Debugger report.

**Why not Clarify:** SageMaker Clarify processing is in maintenance mode and rejects `CreateProcessingJob` for new accounts. SHAP is computed here instead with XGBoost's built-in exact TreeSHAP (`pred_contribs=True`) on the registered model artifact - no job, endpoint or model is created, so this notebook is not billable. SHAP on test windows from seen subjects is still optimistic.

In [ ]:
# SageMaker Python SDK v3 (default on current Studio images) removed the v2 modules this notebook uses.
# Pin v2; on first run this installs it and restarts the kernel - then run the notebook again from the top.
from importlib.metadata import version, PackageNotFoundError
try:
    _sm_ver = version("sagemaker")
except PackageNotFoundError:
    _sm_ver = "0"
if int(_sm_ver.split(".")[0]) != 2:
    %pip uninstall -q -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
    %pip install -q "sagemaker>=2.200,<3"
    print(f"Replaced sagemaker {_sm_ver} with v2 - restarting kernel; re-run the notebook from the top.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("sagemaker", _sm_ver)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json, time
import boto3, numpy as np, pandas as pd
from botocore.exceptions import ClientError
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
from src.pamap2 import monitoring as M
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
cw = session.client("cloudwatch")
sm = session.client("sagemaker")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")
MON = lambda key: s3_uri(cfg, "monitoring", key)
NS = cfg["monitoring_namespace"]

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_bytes(uri):
    b, k = split_key(uri)
    return s3.get_object(Bucket=b, Key=k)["Body"].read()
def read_bytes_or_none(uri):
    try:
        return read_bytes(uri)
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise
def read_parquet(name):
    return pd.read_parquet(io.BytesIO(read_bytes(SPLITS + name)))
def read_json(uri):
    return json.loads(read_bytes(uri))
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)
def put_text(uri, text):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=text.encode())
    return uri
def exists(uri):
    b, k = split_key(uri)
    try:
        s3.head_object(Bucket=b, Key=k)
        return True
    except ClientError:
        return False
def latest_subprefix(uri_prefix):
    """Newest child 'folder' (timestamped run folders sort lexicographically)."""
    b, k = split_key(uri_prefix)
    r = s3.list_objects_v2(Bucket=b, Prefix=k, Delimiter="/")
    subs = sorted(p["Prefix"] for p in r.get("CommonPrefixes", []))
    return f"s3://{b}/{subs[-1]}" if subs else None
def publish(metrics):
    """metrics: {name: value} -> custom namespace (put_metric_data)."""
    cw.put_metric_data(Namespace=NS, MetricData=[{"MetricName": n, "Value": float(v), "Unit": "None"} for n, v in metrics.items()])
    print("Published", metrics)

def safe_publish(metrics):
    try:
        publish(metrics)
    except Exception as e:   # never mask the original error
        print("WARNING: publish failed:", e)
def report_failure(job_name):
    try:
        print("FailureReason:", sm.describe_processing_job(ProcessingJobName=job_name).get("FailureReason"))
    except Exception as e:
        print("Could not describe job:", e)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
ACTIVITY_IDS = list(LABEL_MAP.values())
print(K, "classes;", len(FEATURES), "features")


In [ ]:
# Local exact TreeSHAP needs the xgboost library matching the training container (cfg["xgboost_version"], e.g. 1.7-1).
XGB_PIN = "xgboost==" + cfg["xgboost_version"].split("-")[0] + ".*"
try:
    import xgboost as xgb
except ImportError:
    %pip install -q "{XGB_PIN}"
    import xgboost as xgb
print("xgboost", xgb.__version__)
train_info = read_json(REPORTS + "xgboost_training.json")
PKG_ARN = read_json(REPORTS + "xgboost_eval.json")["model_package_arn"]   # the evaluated package
SHAP_OUT = MON("reports/shap/")

## SHAP explainability (local exact TreeSHAP - not billable)

In [ ]:
import pickle, tarfile
test = read_parquet("test.parquet").reset_index(drop=True)
sample = test.sample(n=min(200, len(test)), random_state=0)

# Model artifact of the evaluated package -> Booster
model_data = sm.describe_model_package(ModelPackageName=PKG_ARN)["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]
with tarfile.open(fileobj=io.BytesIO(read_bytes(model_data)), mode="r:gz") as tf:
    raw = tf.extractfile(next(m for m in tf.getmembers() if m.isfile())).read()
booster = xgb.Booster()
try:
    booster.load_model(bytearray(raw))   # built-in algorithm's native format
except xgb.core.XGBoostError:
    booster = pickle.loads(raw)          # older containers pickle the Booster
print("Model:", model_data)

# Exact TreeSHAP; multiclass -> (n, K, F+1), last column is the bias term. NaNs use the model's missing-value paths.
contribs = booster.predict(xgb.DMatrix(sample[FEATURES].to_numpy(float)), pred_contribs=True)
assert contribs.shape == (len(sample), K, len(FEATURES) + 1), contribs.shape
mean_abs = np.abs(contribs[:, :, :-1]).mean(axis=0)               # (K, F)
class_names = [ACTIVITY_NAMES[LABEL_MAP[i]] for i in range(K)]
global_imp = pd.Series(mean_abs.mean(axis=0), index=FEATURES).sort_values(ascending=False)
per_class = pd.DataFrame(mean_abs.T, index=FEATURES, columns=class_names)

run = SHAP_OUT + time.strftime("%Y%m%d-%H%M%S") + "/"
write_json(run + "analysis.json", {
    "method": "xgboost exact TreeSHAP (pred_contribs)", "model_package_arn": PKG_ARN, "model_data": model_data,
    "n_samples": len(sample), "agg_method": "mean_abs",
    "global_shap_values": global_imp.to_dict(),
    "per_class_shap_values": {c: per_class[c].to_dict() for c in class_names}})
top = per_class.loc[global_imp.index[:20]].assign(overall=global_imp[:20])
put_text(run + "report.html", "<html><head><meta charset='utf-8'><title>SHAP report</title></head><body>"
         f"<h1>SHAP explainability (mean |SHAP|, {len(sample)} test windows)</h1>"
         f"<p>Exact TreeSHAP on {PKG_ARN}</p>" + top.round(4).to_html() + "</body></html>")
print("SHAP output:", run)
display(top.round(4))
global_imp[:20][::-1].plot.barh(figsize=(7, 6), title="Top 20 features - mean |SHAP| over classes");

## Consolidated report index

In [ ]:
dq, mq = latest_subprefix(MON("data-quality/runs/")), latest_subprefix(MON("model-quality/runs/"))
shap_run = latest_subprefix(SHAP_OUT)
debugger = train_info.get("debugger_artifacts")
cands = {
    "data_quality_baseline_statistics": MON("data-quality/baseline/statistics.json"),
    "data_quality_baseline_constraints": MON("data-quality/baseline/constraints.json"),
    "data_quality_violations": dq and dq + "constraint_violations.json",
    "data_quality_violations_summary": dq and dq + "violations_summary.json",
    "model_quality_baseline_statistics": MON("model-quality/baseline/statistics.json"),
    "model_quality_baseline_constraints": MON("model-quality/baseline/constraints.json"),
    "model_quality_violations": mq and mq + "constraint_violations.json",
    "model_quality_violations_summary": mq and mq + "violations_summary.json",
    "shap_analysis": shap_run and shap_run + "analysis.json",
    "shap_report_html": shap_run and shap_run + "report.html",
    "debugger_report": debugger and debugger + M.DEBUGGER_REPORT_PATH,
}
# a violations file legitimately absent means zero violations; keep the summary as the evidence
cands = {k: (v if v and exists(v) else None) for k, v in cands.items()}
index = M.report_index(cands)
write_json(MON("reports/index.json"), index)
display(pd.DataFrame([{"report": k, "uri": v, "exists": exists(v)} for k, v in index["reports"].items()]))
print("Not found (job did not emit them or has not run):", index["missing"])


## Cleanup

In [ ]:
# Cleanup: nothing here creates jobs, schedules or endpoints; list them, delete only temporary pamap2-har-mon* models.
for page in sm.get_paginator("list_monitoring_schedules").paginate(NameContains=cfg["prefix"]):
    for x in page["MonitoringScheduleSummaries"]:
        print("WARNING: monitoring schedule exists (not deleted):", x["MonitoringScheduleName"])
for page in sm.get_paginator("list_models").paginate(NameContains=f"{cfg['prefix']}-mon"):
    for m in page["Models"]:
        if m["ModelName"].startswith(f"{cfg['prefix']}-mon"):
            print("Deleting temporary model", m["ModelName"])
            sm.delete_model(ModelName=m["ModelName"])
for kind, pager, key, field in (("processing", "list_processing_jobs", "ProcessingJobSummaries", "ProcessingJobName"),
                                ("transform", "list_transform_jobs", "TransformJobSummaries", "TransformJobName")):
    for page in sm.get_paginator(pager).paginate(StatusEquals="InProgress", NameContains=cfg["prefix"]):
        for x in page[key]:
            print("WARNING: still running", kind, "job:", x[field])
print("Endpoints:", [e["EndpointName"] for p in sm.get_paginator("list_endpoints").paginate() for e in p["Endpoints"]])
